## Example 1: Basic File Type Detection

In [ ]:
import magic
import tempfile

# Example 1: Basic file type detection
print("Testing python-magic file type detection:")

# Create test files
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write("Hello, this is a plain text file.")
    txt_file = f.name

with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
    f.write("#!/usr/bin/env python3\nprint('Hello')")
    py_file = f.name

# Test from_file() function
print(f"✓ Text file: {magic.from_file(txt_file)}")
print(f"✓ Python file: {magic.from_file(py_file)}")

# Test Magic instance
m = magic.Magic()
print(f"✓ Magic instance: {m.from_file(txt_file)}")

assert magic.from_file(txt_file, mime=True) == "text/plain"
assert "ASCII text" in magic.from_file(txt_file)
assert "python" in magic.from_file(py_file, mime=True)   # text/x-script.python (text/x-python on older libmagic)
assert m.from_file(txt_file) == magic.from_file(txt_file)

# Cleanup
import os
os.unlink(txt_file)
os.unlink(py_file)

## Example 2: MIME Type Detection

In [ ]:
import magic
import tempfile
import os

# Example 2: MIME type detection
print("Testing python-magic MIME type detection:")

# Test from_file with mime=True
with tempfile.NamedTemporaryFile(mode='w', suffix='.html', delete=False) as f:
    f.write('<!DOCTYPE html><html><body>Hello</body></html>')
    html_file = f.name

print(f"✓ Standard: {magic.from_file(html_file)}")
print(f"✓ MIME: {magic.from_file(html_file, mime=True)}")

# Test from_buffer with mime detection
json_buffer = b'{"key": "value"}'
print(f"✓ Buffer type: {magic.from_buffer(json_buffer)}")
print(f"✓ Buffer MIME: {magic.from_buffer(json_buffer, mime=True)}")

# Test Magic instance with mime=True
mime_magic = magic.Magic(mime=True)
print(f"✓ Magic(mime=True): {mime_magic.from_file(html_file)}")

assert magic.from_file(html_file, mime=True) == "text/html"
assert mime_magic.from_file(html_file) == "text/html"
assert magic.from_buffer(json_buffer, mime=True) == "application/json"

# Cleanup
os.unlink(html_file)

## Example 3: Binary File Analysis

In [ ]:
import magic
import tempfile
import os

# Example 3: Binary file detection
print("Testing python-magic binary file detection:")

# Create binary files with magic signatures
with tempfile.NamedTemporaryFile(mode='wb', delete=False) as f:
    f.write(b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR')  # PNG header
    png_file = f.name

with tempfile.NamedTemporaryFile(mode='wb', delete=False) as f:
    f.write(b'%PDF-1.4\n%\xe2\xe3\xcf\xd3')  # PDF header
    pdf_file = f.name

# Detect binary formats
print(f"✓ PNG: {magic.from_file(png_file)}")
print(f"✓ PNG MIME: {magic.from_file(png_file, mime=True)}")
print(f"✓ PDF: {magic.from_file(pdf_file)}")
print(f"✓ PDF MIME: {magic.from_file(pdf_file, mime=True)}")

# Test from_buffer on binary data
print(f"✓ Buffer: {magic.from_buffer(b'\xff\xd8\xff\xe0JFIF')}")

assert magic.from_file(png_file, mime=True) == "image/png"
assert magic.from_file(pdf_file, mime=True) == "application/pdf"
assert magic.from_file(pdf_file).startswith("PDF document, version 1.4")
assert magic.from_buffer(b'\xff\xd8\xff\xe0JFIF').startswith("JPEG image data")

# Cleanup
os.unlink(png_file)
os.unlink(pdf_file)

## Example 4: Encoding Detection

In [ ]:
import magic
import tempfile
import os

# Example 4: Encoding detection
print("Testing python-magic encoding detection:")

# Create files with different encodings
with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', delete=False) as f:
    f.write("Hello World 🌍 café résumé")
    utf8_file = f.name

with tempfile.NamedTemporaryFile(mode='w', encoding='ascii', delete=False) as f:
    f.write("Simple ASCII text")
    ascii_file = f.name

# Test mime_encoding=True
encoding_magic = magic.Magic(mime_encoding=True)
print(f"✓ UTF-8 encoding: {encoding_magic.from_file(utf8_file)}")
print(f"✓ ASCII encoding: {encoding_magic.from_file(ascii_file)}")

# Test buffer encoding
print(f"✓ Buffer encoding: {encoding_magic.from_buffer(b'Plain text')}")

assert encoding_magic.from_file(utf8_file) == "utf-8"
assert encoding_magic.from_file(ascii_file) == "us-ascii"
assert encoding_magic.from_buffer(b'Plain text') == "us-ascii"

# Cleanup
os.unlink(utf8_file)
os.unlink(ascii_file)

## Example 5: File Descriptor Detection

In [ ]:
import magic
import tempfile
import os

# Example 5: File descriptor detection
print("Testing python-magic file descriptor detection:")

# Create test file
with tempfile.NamedTemporaryFile(mode='w', delete=False) as f:
    f.write("#!/bin/bash\necho 'Shell script'")
    script_file = f.name

# Test from_descriptor()
with open(script_file, 'rb') as f:
    print(f"✓ from_descriptor: {magic.from_descriptor(f.fileno())}")
    print(f"✓ Descriptor MIME: {magic.from_descriptor(f.fileno(), mime=True)}")

# Test Magic instance with descriptor
m = magic.Magic()
with open(script_file, 'rb') as f:
    print(f"✓ Magic instance: {m.from_descriptor(f.fileno())}")

with open(script_file, 'rb') as f:
    assert magic.from_descriptor(f.fileno(), mime=True) == "text/x-shellscript"
    assert "shell script" in magic.from_descriptor(f.fileno())

# A ZIP archive written with the zipfile module
import zipfile
with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as f:
    zip_file = f.name
with zipfile.ZipFile(zip_file, 'w') as zf:
    zf.writestr('hello.txt', 'Hello')

with open(zip_file, 'rb') as f:
    print(f"✓ ZIP file: {magic.from_descriptor(f.fileno())}")
    assert magic.from_descriptor(f.fileno(), mime=True) == "application/zip"

# Cleanup
os.unlink(script_file)
os.unlink(zip_file)